### PSMNIST Classification with S4


In [2]:
from __future__ import annotations
from typing import Dict, Any

import torch
import os
from pathlib import Path

from psmnist_task import PSMNISTTask
from src.train_utils.trainer import Trainer
from src.models.v2.build_model import BlockConfig

## Configuration Setup

Define the block configuration constructor that creates model blocks.
This function is **model-agnostic** - just change `kind="s4"` to `kind="lmu"`
to switch between architectures while keeping everything else identical.


In [3]:
def create_block_cfg_ctor(
    kind: str,
    dropout: float,
    mlp_ratio: float,
    droppath_final: float,
    layerscale_init: float,
    residual_gain: float,
    pool: str,
    **kwargs
):
    def block_cfg_ctor(theta: int) -> BlockConfig:
        return BlockConfig(
            kind=kind,
            memory_size=256 if kind == "lmu" else 0,
            theta=theta,
            dropout=dropout,
            mlp_ratio=mlp_ratio,
            droppath_final=droppath_final,
            layerscale_init=layerscale_init,
            residual_gain=residual_gain,
            pool=pool,
            **kwargs
        )
    return block_cfg_ctor


## Hyperparameters & Data Paths

In [4]:
# Setup paths
current_dir = Path.cwd()
project_root = current_dir.parent.parent.parent
data_root = str(project_root / "src" / "datasets" / "psmnist" / "data")

args: Dict[str, Any] = {
    # ==================== DATA ====================
    "data_root": data_root,
    "batch": 128,  # ⬆️ Large batch for efficiency (short sequences)
    "data_loader_kwargs": {
        "num_workers": 0,           # ⚠️ Keep 0 for MPS compatibility
        "permutation_seed": 42,     # Fixed permutation for reproducibility
        "normalize": "standard",    # MNIST standard normalization
        "pin_memory": False,        # Not needed for MPS
        "persistent_workers": False,
    },

    # ==================== TRAINING ====================
    "epochs": 50,           # ⬇️ Quick convergence expected
    "lr": 1e-3,            # Standard learning rate
    "wd": 1e-4,            # Weight decay for regularization
    "amp": True,           # Mixed precision for speed
    "save_dir": "./runs/psmnist_s4_task",
    "warmup_epochs": 5,    # Learning rate warmup
    "patience": 5,         # ⬇️ Fast early stopping
    "min_delta": 0.001,    # Minimum improvement threshold
    "early_key": "accuracy",  # Monitor accuracy for early stopping

    # ==================== MODEL ====================
    "d_model": 128,            # Model dimension
    "depth": 2,                # Number of S4 layers
    "dropout": 0.1,            # ⬇️ Light dropout
    "mlp_ratio": 2.0,          # MLP expansion ratio
    "droppath_final": 0.0,     # ⬇️ No droppath (not needed)
    "layerscale_init": 0.0,    # ⬇️ No layerscale (simpler)
    "residual_gain": 1.0,      # Residual connection scaling
    "pool": "mean",            # Mean pooling over sequence
}

print(f"Data root: {data_root}")
print(f"Save directory: {args['save_dir']}")


📁 Data root: /Users/glbrlb/PycharmProjects/Msc/LMU_S4/src/datasets/psmnist/data
💾 Save directory: ./runs/psmnist_s4_task


## Model Setup & Device Selection

Create the S4 block configuration constructor and select the appropriate device.
The configuration is **model-agnostic** - only the `kind="s4"` parameter
makes this an S4 model instead of LMU.


In [5]:
# Create S4 block configuration constructor
args["block_cfg_ctor"] = create_block_cfg_ctor(
    kind="s4",
    dropout=args["dropout"],
    mlp_ratio=args["mlp_ratio"],
    droppath_final=args["droppath_final"],
    layerscale_init=args["layerscale_init"],
    residual_gain=args["residual_gain"],
    pool=args["pool"],
)

# Device selection with MPS optimization for M-series Macs
if torch.backends.mps.is_available():
    args["device"] = torch.device("mps")
    print("🚀 Using MPS (Apple Silicon)")
elif torch.cuda.is_available():
    args["device"] = torch.device("cuda")
    print("🚀 Using CUDA")
else:
    args["device"] = torch.device("cpu")
    args["amp"] = False  # Disable AMP for CPU
    print("Using CPU (slower)")

🚀 Using MPS (Apple Silicon)


## Training

Initialize the task, apply MPS optimizations if available, and train the model.
The `Trainer` handles:
- Model building from the block config
- Training loop with progress bars
- Validation and early stopping
- Checkpoint saving
- Mixed precision training (AMP)


In [ ]:
task = PSMNISTTask()

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)
    os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

print("\n" + "="*60)
print("INITIALIZING S4 MODEL FOR PSMNIST")
print("="*60)

trainer = Trainer(args=args, task=task)

print("\n🚀 Starting training...")
best_metric, ckpt_path = trainer.fit()

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"✅ Best {trainer.early_key}: {best_metric:.4f}")
print(f"💾 Checkpoint saved: {ckpt_path}")
print("="*60 + "\n")

In [6]:
import matplotlib.pyplot as plt

def plot_history(history):

    plt.figure(figsize=(12, 5))

    # Plot training and validation loss
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()

    # Plot training and validation accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Val Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
from src.utils.checkpoint import load_trainer_from_checkpoint

trainer = load_trainer_from_checkpoint(
    checkpoint_path=args["save_dir"] + "/best.pt",
    args=args,
    task=PSMNISTTask(),
)

history = trainer.history

plot_history(history)

## Evaluation

Load the best checkpoint and evaluate on the test set.
This provides the final accuracy metric for the S4 model on PSMNIST.


In [ ]:
from psmnist_eval import evaluate_best_model, print_evaluation_results

# Evaluate best model on test set
print("🔍 Evaluating best model on test set...")
logits_test, labels_test = evaluate_best_model(
    args=args,
    task=PSMNISTTask(),
    best_model_path=f"{args['save_dir']}/best.pt",
)

# Print results
print_evaluation_results(
    logits_test=logits_test,
    labels_test=labels_test,
)


### Changes in dataset length

In [ ]:
from src.train_utils.trainer import Trainer
import matplotlib.pyplot as plt
from src.notebooks.psmnist.psmnist_eval import evaluate_best_model, print_evaluation_results

if torch.backends.mps.is_available():
    torch.mps.set_per_process_memory_fraction(0.9)
    os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"


for frac in [0.25, 0.5]:
    print(f"\n🔄 Training with fraction: {frac}")

    args["fraction"] = frac
    args["save_dir"] = f"./runs/psmnist_s4_task_frac_{frac}"
    trainer = Trainer(args=args, task=PSMNISTTask())
    best_metric, best_path = trainer.fit()

    print(f"\n✅ Training complete for fraction {frac}! Best validation {trainer.early_key}: {best_metric:.4f}")
    print(f"💾 Best model saved to: {best_path}")

    history = trainer.history

    plot_history(history)

    logits_test, labels_test = evaluate_best_model(
        args=args,
        task=PSMNISTTask(),
        best_model_path=best_path
    )

    print_evaluation_results(
    logits_test=logits_test,
    labels_test=labels_test,
)